In [50]:
from textblob import TextBlob
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from textblob import TextBlob
from scipy.stats import spearmanr, kendalltau

In [139]:
associations_valid = [('data/test_data/few_shot/test_data_2024_abstract_prompts_fewshot.jsonl', 'results/gpt4o/test_data_answer_not_in_prompt/test_data_2024_abstract_prompts_fewshot_gpt4o_results.jsonl', 'GPT4o-Abstract-Fewshot'),
                    #   ('data/test_data/few_shot/test_data_2024_abstract_prompts_fewshot.jsonl', 'results/gpt4o/test_data_answer_not_in_prompt/test_data_2024_abstract_prompts_oneshot_gpt4o_results.jsonl', 'GPT4o-Abstract-Oneshot'),
                    #   ('data/test_data/few_shot/test_data_2024_abstract_prompts_fewshot.jsonl', 'results/gpt4o/test_data_answer_not_in_prompt/test_data_2024_abstract_prompts_gpt4o_results.jsonl', 'GPT4o-Abstract-Zeroshot'),
                    #   ('data/test_data/few_shot/test_data_2024_summary_prompts_fewshot.jsonl', 'results/gpt4o/test_data_answer_not_in_prompt/test_data_2024_summary_prompts_fewshot_gpt4o_results.jsonl', 'GPT4o-Summary-Fewshot'),
                    #   ('data/test_data/few_shot/test_data_2024_summary_prompts_fewshot.jsonl', 'results/gpt4o/test_data_answer_not_in_prompt/test_data_2024_summary_prompts_oneshot_gpt4o_results.jsonl', 'GPT4o-Summary-Oneshot'),
                    #   ('data/test_data/few_shot/test_data_2024_summary_prompts_fewshot.jsonl', 'results/gpt4o/test_data_answer_not_in_prompt/test_data_2024_summary_prompts_gpt4o_results.jsonl', 'GPT4o-Summary-Zeroshot'),
                    #   ('data/test_data/few_shot/test_data_2024_summary_prompts_fewshot.jsonl', 'results/gpt4o/test_data_answer_not_in_prompt/test_data_2024_full text_prompts_oneshot_gpt4o_results.jsonl', 'GPT4o-Full_Text-Oneshot'),
                    #   ('data/test_data/few_shot/test_data_2024_summary_prompts_fewshot.jsonl', 'results/gpt4o/test_data_answer_not_in_prompt/test_data_2024_full text_prompts_gpt4o_results.jsonl', 'GPT4o-Full_Text-Zeroshot'),
                      
                      ('data/test_data/few_shot/test_data_2024_abstract_prompts_fewshot.jsonl', 'results/llama3_8BInstruct/few_shot_2024_summary_prompts.jsonl', 'Llama-Summary-Zeroshot'),
                      ('data/test_data/few_shot/test_data_2024_abstract_prompts_fewshot.jsonl', 'results/llama3_8BInstruct/zero_shot_2024_summary_prompts.jsonl', 'Llama-Summary-Zeroshot'),
                      ('data/test_data/few_shot/test_data_2024_abstract_prompts_fewshot.jsonl', 'results/llama3_8BInstruct/zero_shot_2024_abstract_prompts.jsonl', 'Llama-Abstract-Zeroshot'),]


In [177]:
with open(associations_valid[1][1], 'r') as f:
    data = [json.loads(line) for line in f]

In [172]:
data[1]['review']

'{\n  "Soundness": 4, "Presentation": 4, "Contribution": 3, "Rating": 9, "Confidence": 4, "Strengths": "The Bradley-Terry loss is a good alternative to MSE for learning sparse latent fitness functions from globally epistatic data. The authors demonstrate its effectiveness on various simulated and real-world datasets. The paper\'s contribution to the field of protein engineering is significant, as it provides a more data-efficient and robust method for predicting fitness functions.\\nThe use of contrastive losses to learn fitness functions is an interesting direction for future research. The authors\' experiments on the FLIP benchmark tasks are thorough and well-justified. The paper is well-written and easy to follow.", "Weaknesses": "The authors\' claim that contrastive losses bypass the issues caused by nonlinearity is not entirely clear. The paper could benefit from a more detailed explanation of how the Bradley-Terry loss handles nonlinearity in the latent fitness functions. The aut

In [173]:
json.loads(data[0]['review'])

{'Soundness': 4,
 'Presentation': 4,
 'Contribution': 4,
 'Rating': 9,
 'Confidence': 5,
 'Strengths': 'The authors have successfully integrated weight-entanglement and gradient-based NAS methods, which is a significant contribution. The proposed TangleNAS approach is efficient and accurate, and it has the potential to revolutionize the field of neural architecture search. The paper is well-written and easy to follow.',
 'Weaknesses': 'The paper assumes that the reader is familiar with the basics of NAS and weight-entanglement, which might make it difficult for non-experts to understand. Additionally, the authors could have provided more detailed experimental results and comparisons with other state-of-the-art methods.',
 'Questions': 'How does TangleNAS handle the issue of overfitting in weight-entangled spaces? Can the authors provide more insights into the impact of the zero-padding operation on the performance of the model? Are there any plans to extend TangleNAS to other types of 

In [178]:
#[{item['paper_id']:json.loads(item['review'])} for item in data if 'review' in item.keys()]
for item in data:
    print(item['review'].strip())
    jsonloads = json.loads(item['review'].strip())

{
  "Soundness": 4,
  "Presentation": 4,
  "Contribution": 4,
  "Rating": 9,
  "Confidence": 5,
  "Strengths": "The authors have successfully integrated weight-entanglement and gradient-based NAS methods, which is a significant contribution. The proposed TangleNAS approach is efficient and accurate, and it has the potential to revolutionize the field of neural architecture search. The paper is well-written and easy to follow.",
  "Weaknesses": "The paper assumes that the reader is familiar with the basics of NAS and weight-entanglement, which might make it difficult for non-experts to understand. Additionally, the authors could have provided more detailed experimental results and comparisons with other state-of-the-art methods.",
  "Questions": "How does TangleNAS handle the issue of overfitting in weight-entangled spaces? Can the authors provide more insights into the impact of the zero-padding operation on the performance of the model? Are there any plans to extend TangleNAS to other

JSONDecodeError: Unterminated string starting at: line 1 column 1220 (char 1219)

In [131]:
def get_gold_and_predictions(association):
    with open(association[0], 'r') as f:
        gold_standard = [json.loads(line) for line in f]
        #gold_standard = json.load(f)

    with open(association[1], 'r') as f:
        predictions = [json.loads(line) for line in f]

    gold_standard = [{item['paper_id']:json.loads(item['messages'][-1]['content'])} for item in gold_standard]
    prediction = [{item['paper_id']:item['review']} for item in predictions if 'review' in item.keys()]
    if isinstance(next(iter(prediction[0].values())), str):
        prediction = [{item['paper_id']:json.loads(item['review'])}  for item in predictions if 'review' in item.keys()]
    
    return gold_standard, prediction

In [132]:
with open(associations_valid[1][1], 'r') as f:
    predictions = [json.loads(line) for line in f]
predictions[0]['review']
[{item['paper_id']:json.loads(item['review'])} for item in predictions if 'review' in item.keys()]

[{'B0OwtVEejJ': {'Soundness': 4,
   'Presentation': 3,
   'Contribution': 4,
   'Rating': 9,
   'Confidence': 5,
   'Strengths': 'The authors provide a clear and well-structured paper with a good hypothesis and a well-defined method. The experiments are comprehensive and the results are presented in a clear and concise manner.',
   'Weaknesses': 'The paper assumes some knowledge of NAS methods and weight-entanglement, which might make it difficult for non-experts to understand. The authors could provide more details about the implementation of TangleNAS.',
   'Questions': 'How does TangleNAS compare to other NAS methods in terms of computational complexity? Are there any potential drawbacks to using weight-entanglement in larger search spaces?'}},
 {'ZlEtXIxl3q': {'Soundness': 4,
   'Presentation': 3,
   'Contribution': 4,
   'Rating': 9,
   'Confidence': 5,
   'Strengths': 'Effective use of contrastive losses, improvement over existing methods, data-efficiency and robustness.',
   'We

In [133]:
def score_results(scores):
    scores_dict = {}
    for score in scores:
        tot_score = 0
        for key in ['Soundness', 'Presentation', 'Contribution', 'Rating', 'Confidence']:
            tot_score += list(score.values())[0][key]
        tot_score += (TextBlob(list(score.values())[0]['Strengths']).sentiment.polarity - np.abs(TextBlob(list(score.values())[0]['Weaknesses']).sentiment.polarity))
        scores_dict[list(score.keys())[0]] = tot_score
    sorted_items = sorted(scores_dict.items(), key=lambda item: item[1], reverse=True)
    ranked_values = {}
    rank = 1
    for key, value in sorted_items:
        ranked_values[key] = rank
        rank += 1
    return ranked_values

In [140]:
for association in associations_valid:
    gold_standard, predictions = get_gold_and_predictions(association)
    gold_scores = score_results(gold_standard)
    pred_scores = score_results(predictions)
    gold_rank = []
    pred_rank = []
    standard_keys = [list(item.keys())[0] for item in gold_standard]
    for key in standard_keys:
        if key in pred_scores.keys():
            gold_rank.append(gold_scores[key])
            pred_rank.append(pred_scores[key])
    print(association[2])
    print(spearmanr(gold_rank, pred_rank))
    print(kendalltau(gold_rank, pred_rank))
    print('\n')

GPT4o-Abstract-Fewshot
SignificanceResult(statistic=0.28926547188697405, pvalue=0.0038662877838358707)
SignificanceResult(statistic=0.2026088786029876, pvalue=0.0031192420025038616)




JSONDecodeError: Unterminated string starting at: line 1 column 1220 (char 1219)

In [115]:
predictions

[]

In [103]:
list(predictions[0].values())[0]

'{\n  "Soundness": 4,\n  "Presentation": 3,\n  "Contribution": 4,\n  "Rating": 9,\n  "Confidence": 4,\n  "Strengths": "The work is well-motivated and aims to bridge the gap between two sub-communities in NAS. The authors have made a significant contribution by adapting gradient-based methods for weight-entangled spaces. The code is openly accessible, which is a plus. The comparative assessment and analysis of performance are thorough and reveal the benefits of the integration.",\n  "Weaknesses": "The paper assumes that the reader is familiar with NAS and weight entanglement, which might make it difficult for some readers to follow. The proposed scheme might not be applicable to all weight-entangled spaces. The evaluation of the proposed method is limited to a single experiment.",\n  "Questions": "How generalizable is the proposed scheme to different weight-entangled spaces? What are the computational costs of adapting gradient-based methods for weight-entangled spaces? How does the pro

In [104]:
list(gold_standard[0].values())[0]

{'Soundness': 2,
 'Presentation': 2,
 'Contribution': 2,
 'Rating': 3,
 'Confidence': 4,
 'Strengths': 'The paper proposes a scheme to adapt gradient-based methods for weight-entangled spaces in neural architecture search (NAS). This integration of weight-entanglement and gradient-based NAS is a new approach that has not been explored before. The paper also presents a comprehensive evaluation of the properties of single and two-stage approaches, including any-time performance, memory consumption, robustness to training fraction, and the effect of fine-tuning.',
 'Weaknesses': '1. The novelty is limited. Some works have also suggested that the weights of large kernel convolution operations can be shared in differentiable neural architecture search. For example, MergeNAS: Merge Operations into One for Differentiable Architecture Search (in IJCAI20)\n2. The performance improvements are limited compared with the baselines.',
 'Questions': '1. How to entangle non-parameter operations, such 

In [78]:
[list(item.keys())[0] for item in gold_standard]

['B0OwtVEejJ',
 'ZlEtXIxl3q',
 '6yJuDK1DsK',
 'lt6xKGGWov',
 'Bvrc6kobWd',
 'FJjHQS2DyE',
 'xelrLobW0n',
 'bYQkOPvgDw',
 'sKPzAXoylB',
 'H8Qg1IIMaR',
 'tth2qXY7RU',
 'fjf3YenThE',
 'H9DYMIpz9c',
 '4kLVvIh8cp',
 'CSpWgKo0ID',
 'gwbQ2YwLhD',
 'eIYDKNqXuV',
 'bAMPOUF227',
 'eNoiRal5xi',
 'uU0Adp7Sfo',
 'dKju7tbe6D',
 'DL7JWbdGr3',
 '1PaDPHDhwe',
 'iI7hZSczxE',
 'qrGjFJVl3m',
 'qup9xD8mW4',
 'Zw8YxUWL4R',
 '3NXhwkZGjz',
 'Fn655mJ4bv',
 'VyMW4YZfw7',
 '09xFexjhqE',
 'nmBjBZoySX',
 'di52zR8xgf',
 '0b328CMwn1',
 'AMDKqZcZbi',
 'am7BPV3Cwo',
 'R1crLHQ4kf',
 'i7P2mK3x3o',
 'YIls9HEa52',
 'koYsgfEwCQ',
 'o6AN2ZoNw2',
 'HXZK1Z8tHa',
 'QO3yH7X8JJ',
 'sNtDKdcI1f',
 'HFtrXBfNru',
 '4u0ruVk749',
 'Pa6SiS66p0',
 'rkplYfqUr0',
 'pTU2X9mUBe',
 'uhtQyRrTzY',
 'WZfatbNdLV',
 'YkRwadXWHd',
 'q4Bim1dDzb',
 '39cPKijBed',
 'aG3EARrrd1',
 'Ts95eXsPBc',
 'LfhG5znxzR',
 'bUGzjiUsIq',
 'uMAujpVi9m',
 'uqxBTcWRnj',
 'o4CLLlIaaH',
 'KNzL1nglNB',
 'pvhyBB86Bt',
 'Mylk5iamJC',
 'MsOcVFzv8D',
 'fBlHaSGKNg',
 'HwQ8NVvd

In [74]:
scores = [{item['paper_id']:json.loads(item['messages'][-1]['content'])} for item in data]
#scores = [{item['paper_id']:item['review']} for item in data if 'review' in item.keys()]
scores

[{'B0OwtVEejJ': {'Soundness': 2,
   'Presentation': 2,
   'Contribution': 2,
   'Rating': 3,
   'Confidence': 4,
   'Strengths': 'The paper proposes a scheme to adapt gradient-based methods for weight-entangled spaces in neural architecture search (NAS). This integration of weight-entanglement and gradient-based NAS is a new approach that has not been explored before. The paper also presents a comprehensive evaluation of the properties of single and two-stage approaches, including any-time performance, memory consumption, robustness to training fraction, and the effect of fine-tuning.',
   'Weaknesses': '1. The novelty is limited. Some works have also suggested that the weights of large kernel convolution operations can be shared in differentiable neural architecture search. For example, MergeNAS: Merge Operations into One for Differentiable Architecture Search (in IJCAI20)\n2. The performance improvements are limited compared with the baselines.',
   'Questions': '1. How to entangle n

In [36]:
list(scores[90].values())[0]

{'Soundness': 4,
 'Presentation': 3,
 'Contribution': 3,
 'Rating': 8,
 'Confidence': 3,
 'Strengths': '1. It is reasonable to utilize the surface where the point clouds should be to compute the distance of two 3D point clouds.\n2.The experiments show the proposed method improves the performance of all the downstream tasks.\n3. The implementation of methods of all the downstream tasks are explained in detail.',
 'Weaknesses': 'The generation of reference points are not explained very clearly. The reviewer is confused by the shared identical weight operation and the reference point generation process.',
 'Questions': 'The reference points are generated from one of the two point clouds, and the weights in Equ.1 are computed for each kNN points and reference points. The reviewer wants to ask how to share the weights in g(q_m, P1) and g(q_m, P2)?'}

In [39]:
TextBlob(list(scores[90].values())[0]['Questions']).sentiment.polarity 

0.2

In [30]:
scores_dict = {}
for score in scores:
    tot_score = 0
    for key in ['Soundness', 'Presentation', 'Contribution', 'Rating', 'Confidence']:
        tot_score += list(score.values())[0][key]
    tot_score += (TextBlob(list(score.values())[0]['Strengths']).sentiment.polarity - np.abs(TextBlob(list(score.values())[0]['Weaknesses']).sentiment.polarity))
    scores_dict[list(score.keys())[0]] = tot_score

In [31]:
scores_dict

{'B0OwtVEejJ': 13.008658008658008,
 'ZlEtXIxl3q': 17.151190476190475,
 '6yJuDK1DsK': 12.848214285714286,
 'lt6xKGGWov': 12.904285714285715,
 'Bvrc6kobWd': 10.341666666666667,
 'FJjHQS2DyE': 16.98611111111111,
 'xelrLobW0n': 13.106468253968254,
 'bYQkOPvgDw': 15.003875968992249,
 'sKPzAXoylB': 17.104005480335267,
 'H8Qg1IIMaR': 14.8625,
 'tth2qXY7RU': 13.401984126984127,
 'fjf3YenThE': 10.989423076923076,
 'H9DYMIpz9c': 18.107323232323232,
 '4kLVvIh8cp': 14.909954545454546,
 'CSpWgKo0ID': 13.836666666666666,
 'gwbQ2YwLhD': 15.6125,
 'eIYDKNqXuV': 14.38088578088578,
 'bAMPOUF227': 17.238247863247864,
 'eNoiRal5xi': 16.873333333333335,
 'uU0Adp7Sfo': 18.246102092352093,
 'dKju7tbe6D': 16.054698581560285,
 'DL7JWbdGr3': 15.140884494293585,
 '1PaDPHDhwe': 10.256578947368421,
 'iI7hZSczxE': 8.215669642857144,
 'qrGjFJVl3m': 16.044078621031748,
 'qup9xD8mW4': 18.00476827094474,
 'Zw8YxUWL4R': 17.311988304093568,
 '3NXhwkZGjz': 16.46203703703704,
 'Fn655mJ4bv': 18.252920227920228,
 'VyMW4YZfw7

In [53]:
sorted_items = sorted(scores_dict.items(), key=lambda item: item[1], reverse=True)
sorted_items

[('Ts95eXsPBc', 23.103410998723497),
 ('uhtQyRrTzY', 23.048214285714288),
 ('Fq8tKtjACC', 22.899333333333335),
 ('uqxBTcWRnj', 21.292045454545455),
 ('kXHEBK9uAY', 21.097278911564626),
 ('lEkFq4RUCX', 21.065),
 ('di52zR8xgf', 20.33),
 ('koYsgfEwCQ', 20.223669467787115),
 ('RlfD5cE1ep', 20.20051984126984),
 ('QO3yH7X8JJ', 19.21212121212121),
 ('MsOcVFzv8D', 19.143177083333335),
 ('sNtDKdcI1f', 18.939583333333335),
 ('ttRSBZiCiu', 18.261240842490842),
 ('Fn655mJ4bv', 18.252920227920228),
 ('uU0Adp7Sfo', 18.246102092352093),
 ('AMDKqZcZbi', 18.179657369146007),
 ('H9DYMIpz9c', 18.107323232323232),
 ('R1crLHQ4kf', 18.09664502164502),
 ('nmBjBZoySX', 18.068571428571428),
 ('YCPDFfmkFr', 18.028787878787877),
 ('am7BPV3Cwo', 18.0259555984556),
 ('qup9xD8mW4', 18.00476827094474),
 ('rkplYfqUr0', 17.93722222222222),
 ('RtDok9eS3s', 17.914345238095237),
 ('Yp01vcQSNl', 17.388939393939395),
 ('Zw8YxUWL4R', 17.311988304093568),
 ('bAMPOUF227', 17.238247863247864),
 ('WZfatbNdLV', 17.16257521124708

In [54]:
{k: rank + 1 for rank, (k, v) in enumerate(sorted_items)}

{'Ts95eXsPBc': 1,
 'uhtQyRrTzY': 2,
 'Fq8tKtjACC': 3,
 'uqxBTcWRnj': 4,
 'kXHEBK9uAY': 5,
 'lEkFq4RUCX': 6,
 'di52zR8xgf': 7,
 'koYsgfEwCQ': 8,
 'RlfD5cE1ep': 9,
 'QO3yH7X8JJ': 10,
 'MsOcVFzv8D': 11,
 'sNtDKdcI1f': 12,
 'ttRSBZiCiu': 13,
 'Fn655mJ4bv': 14,
 'uU0Adp7Sfo': 15,
 'AMDKqZcZbi': 16,
 'H9DYMIpz9c': 17,
 'R1crLHQ4kf': 18,
 'nmBjBZoySX': 19,
 'YCPDFfmkFr': 20,
 'am7BPV3Cwo': 21,
 'qup9xD8mW4': 22,
 'rkplYfqUr0': 23,
 'RtDok9eS3s': 24,
 'Yp01vcQSNl': 25,
 'Zw8YxUWL4R': 26,
 'bAMPOUF227': 27,
 'WZfatbNdLV': 28,
 'ZlEtXIxl3q': 29,
 'sKPzAXoylB': 30,
 'jHdz0CIS2y': 31,
 'YkRwadXWHd': 32,
 'pvhyBB86Bt': 33,
 'uMAujpVi9m': 34,
 'Mdk7YP52V3': 35,
 'i7LCsDMcZ4': 36,
 'hRos9WldRK': 37,
 'FJjHQS2DyE': 38,
 '8fQlGQkj0S': 39,
 'eNoiRal5xi': 40,
 '3NXhwkZGjz': 41,
 '39cPKijBed': 42,
 '8JCn0kmS8W': 43,
 'fQHb1uZzl7': 44,
 'LndMyiBl3n': 45,
 'Pa6SiS66p0': 46,
 'q4Bim1dDzb': 47,
 '09xFexjhqE': 48,
 'dKju7tbe6D': 49,
 '0b328CMwn1': 50,
 'qrGjFJVl3m': 51,
 'LWDRiFzbHQ': 52,
 'pTU2X9mUBe': 53,
 '

In [46]:
new_dict = {}
for item in scores:
   name = list(item.keys())[0]
   new_dict[name] = item